# Notebook 06 — Análisis exploratorio para el extractor de recomendación

## Objetivo

Diseñar las reglas y patrones para el módulo `src/extractor_recomendacion.py`, que clasifica las recomendaciones clínicas de los informes mamográficos en categorías estándar. Antes de codificar el módulo, este notebook documenta el análisis del corpus que fundamenta cada decisión de diseño.

## Lo que se decide en este notebook

1. Qué categorías clínicas usar para clasificar las recomendaciones
2. Cuál es la jerarquía de prioridad cuando una recomendación combina varias categorías
3. Qué patrones regex cubren la mayor parte del corpus
4. Qué typos del corpus deben corregirse en el normalizador
5. Cuántos casos quedarán como fallback para la capa de similitud TF-IDF

## Resultado obtenido

- **100% de cobertura regex** sobre el corpus de 4 347 recomendaciones
- **8 categorías clínicas** definidas
- **9 typos identificados** y corregidos por el normalizador
- **Jerarquía clínica** que prioriza dudas diagnósticas activas sobre planes definidos
- **TF-IDF queda como red de seguridad** para vocabulario no contemplado de otros corpus


---

## Paso 1 — Imports y carga del corpus


In [1]:
import pandas as pd
import re
import unicodedata
from collections import Counter

DATA_PATH = "../data/processed/reports_cleaned.csv"
df = pd.read_csv(DATA_PATH)

# Filtrar solo los informes con recomendación poblada
df_rec = df[df["Recommendations"] != "sin_recomendacion"].copy()

print(f"Total informes: {len(df)}")
print(f"Con recomendación poblada: {len(df_rec)} ({100*len(df_rec)/len(df):.1f}%)")
print(f"Con 'sin_recomendacion': {len(df) - len(df_rec)} ({100*(len(df)-len(df_rec))/len(df):.1f}%)")


Total informes: 4357
Con recomendación poblada: 4347 (99.8%)
Con 'sin_recomendacion': 10 (0.2%)


---

## Paso 2 — Exploración inicial: textos únicos y plantillas dominantes

**Hipótesis inicial**: los informes están muy formulados, con plantillas que se repiten frecuentemente. Si la hipótesis es cierta, podremos cubrir la mayor parte del corpus con pocas reglas regex.


In [2]:
print(f"Recomendaciones únicas (texto exacto): {df_rec['Recommendations'].nunique()}")
print(f"De un total de {len(df_rec)} informes")
print(f"Razón de repetición: {len(df_rec) / df_rec['Recommendations'].nunique():.1f}x")

print("\n" + "=" * 70)
print("TOP 10 RECOMENDACIONES MÁS FRECUENTES")
print("=" * 70)
top10 = df_rec["Recommendations"].value_counts().head(10)
for i, (texto, n) in enumerate(top10.items(), 1):
    print(f"\n[{i:2d}] {n:4d} informes ({100*n/len(df_rec):.1f}%)")
    print(f"     {texto[:180]}")


Recomendaciones únicas (texto exacto): 296
De un total de 4347 informes
Razón de repetición: 14.7x

TOP 10 RECOMENDACIONES MÁS FRECUENTES

[ 1]  415 informes (9.5%)
     - Se sugiere control mamográfico anual.

[ 2]  397 informes (9.1%)
     - Se sugiere control anual.

[ 3]  374 informes (8.6%)
     - Se sugiere correlación con ecografía mamaria debido al patrón mamográfico.

[ 4]  368 informes (8.5%)
     - SE SUGIERE CONTROL MAMOGRÁFICO ANUAL.

[ 5]  318 informes (7.3%)
     - SE SUGIERE CORRELACIÓN CON ECOGRAFÍA MAMARIA DEBIDO AL PATRÓN MAMOGRÁFICO.

[ 6]  316 informes (7.3%)
     - Se sugiere ecografía mamaria para posterior recategorización.

[ 7]  310 informes (7.1%)
     - SE SUGIERE ECOGRAFÍA MAMARIA PARA POSTERIOR RECATEGORIZACIÓN.

[ 8]  189 informes (4.3%)
     - Se sugiere correlación con ecografía mamaria y control mamográfico anual.

[ 9]  182 informes (4.2%)
     - Se sugiere correlación con ecografía mamaria para posterior recategorización.

[10]  139 informes (3.2%)
 

**Observación**: confirmamos la hipótesis. 296 textos únicos para 4 347 informes (relación ~15x). Las top 10 plantillas cubren un porcentaje sustancial del corpus, con duplicación significativa por variaciones de mayúsculas/minúsculas (ejemplo: filas 1 y 4 son la misma frase). Esto justifica usar:

- **Normalización** (NFKD + minúsculas) para colapsar mayúsculas/tildes
- **Regex sobre las plantillas dominantes** para clasificar el grueso del corpus


---

## Paso 3 — Definición de categorías clínicas

Tras inspeccionar las plantillas dominantes y consultar lógica clínica, defino **8 categorías** que capturan las acciones clínicamente distintas:

| Categoría | Significado clínico | Acción operativa |
|---|---|---|
| `estudio_complementario_imagen` | Hallazgo requiere otra técnica de imagen | Agendar eco/RM/magnificación |
| `correlacion_ecografica` | Duda diagnóstica → eco complementaria | Agendar eco a corto plazo |
| `comparacion_estudios_previos` | Buscar exámenes anteriores | Acceso al expediente |
| `control_anual` | Seguimiento rutinario | Próxima mamografía en 12 meses |
| `control_corto_plazo` | Vigilancia activa | Próximo estudio en 3-6 meses |
| `biopsia_histologia` | Confirmación tisular | Procedimiento intervencional |
| `criterio_medico` | Delegación de decisión | Manejo individualizado |
| `derivacion_oncologica` | Manejo especializado | Derivación |

Adicionalmente, `ambigua` para casos no clasificables y `no_recomendacion` para informes sin recomendación.


---

## Paso 4 — Jerarquía clínica para resolver múltiples categorías

**Problema**: muchas recomendaciones combinan dos o tres categorías. Cuando se cotejen contra la tabla ACR, necesito decidir cuál es la categoría 'principal'.

**Razonamiento clínico** (principio del peor caso, anteponerse a lo peor):

1. Acciones diagnósticas urgentes priman sobre planes definidos
2. Una solicitud de información adicional (correlación, comparación) señala duda diagnóstica activa, que es más urgente que un control con plazo ya establecido
3. La biopsia es la acción de mayor prioridad porque define diagnóstico
4. Los planes definidos en el tiempo (controles) tienen menor urgencia que la resolución de dudas

**Jerarquía resultante** (de mayor a menor prioridad):

1. `biopsia_histologia`
2. `derivacion_oncologica`
3. `estudio_complementario_imagen`
4. `correlacion_ecografica`
5. `comparacion_estudios_previos`
6. `control_corto_plazo`
7. `control_anual`
8. `criterio_medico`


In [3]:
JERARQUIA_CLINICA = [
    "biopsia_histologia",
    "derivacion_oncologica",
    "estudio_complementario_imagen",
    "correlacion_ecografica",
    "comparacion_estudios_previos",
    "control_corto_plazo",
    "control_anual",
    "criterio_medico",
]

print("Jerarquía clínica definida:")
for i, cat in enumerate(JERARQUIA_CLINICA, 1):
    print(f"  {i}. {cat}")


Jerarquía clínica definida:
  1. biopsia_histologia
  2. derivacion_oncologica
  3. estudio_complementario_imagen
  4. correlacion_ecografica
  5. comparacion_estudios_previos
  6. control_corto_plazo
  7. control_anual
  8. criterio_medico


---

## Paso 5 — Construcción del normalizador de texto

El normalizador opera en dos niveles:

**Nivel 1 — Ortográfico**: NFKD para quitar tildes, minúsculas, colapso de espacios.

**Nivel 2 — Typos clínicos**: diccionario construido con los typos reales que aparecen en el corpus.


In [4]:
# Diccionario de typos identificados durante la exploración
TYPOS_CLINICOS = {
    r"\bcografia\b":         "ecografia",       # COGRAFIA: falta E inicial
    r"\bsuerimos\b":         "sugerimos",       # SUERIMOS: falta G
    r"\bsuegerimos\b":       "sugerimos",       # SUEGERIMOS: G extra
    r"\bmamografica\b":      "mamografico",     # género equivocado
    r"\brecategorizaicon\b": "recategorizacion",# transposición de letras
    r"\bmanual\b":           "anual",           # autocorrector convirtió ANUAL → MANUAL
    r"\banula\b":            "anual",           # orden de letras
    r"\btratatne\b":         "tratante",        # transposición
    r"\bcontro\b":           "control",         # falta L final
}

def normalizar_texto(texto):
    """Normaliza ortografía y corrige typos clínicos conocidos."""
    if not isinstance(texto, str):
        return ""
    # Nivel 1: NFKD + minúsculas
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    texto = texto.lower()
    # Nivel 2: typos
    for typo, correcto in TYPOS_CLINICOS.items():
        texto = re.sub(typo, correcto, texto)
    return texto

# Test del normalizador
ejemplos = [
    "- SE SUGIERE CONTROL MAMOGRÁFICO ANUAL.",
    "- SUERIMOS COMPLEMENTAR CON ECOGRAFÍA MAMARIA.",
    "- SE SUGIERE CONTROL MAMOGRÁFICO MANUAL.",
    "- SE SUGIERE CONTRO ANUAL.",
]
print("Test del normalizador:")
for ej in ejemplos:
    print(f"\n  Original:    {ej}")
    print(f"  Normalizado: {normalizar_texto(ej)}")


Test del normalizador:

  Original:    - SE SUGIERE CONTROL MAMOGRÁFICO ANUAL.
  Normalizado: - se sugiere control mamografico anual.

  Original:    - SUERIMOS COMPLEMENTAR CON ECOGRAFÍA MAMARIA.
  Normalizado: - sugerimos complementar con ecografia mamaria.

  Original:    - SE SUGIERE CONTROL MAMOGRÁFICO MANUAL.
  Normalizado: - se sugiere control mamografico anual.

  Original:    - SE SUGIERE CONTRO ANUAL.
  Normalizado: - se sugiere control anual.


---

## Paso 6 — Patrones regex por categoría

Para cada una de las 8 categorías clínicas defino patrones regex que cubren las variantes vistas en el corpus. Cada patrón se construye sobre el texto YA NORMALIZADO (sin tildes, en minúsculas).


In [5]:
PATRONES_POR_CATEGORIA = {
    "estudio_complementario_imagen": [
        r"ecografia\s+mamaria\s+(\w+\s+)?(actualizada\s+)?para\s+(posterior\s+)?recategorizacion",
        r"ecografia\s+complementaria",
        r"estudio\s+ecografico\s+complementario",
        r"estudio\s+complementario\s+de\s+ecografia",
        r"ecografia\s+mamaria\s+(actualizada\s+)?(debido\s+al|por\s+el)\s+patron",
        r"ecografia\s+mamaria\s+actualizada",
        r"complementar\s+(el\s+estudio\s+)?con\s+(una\s+)?ecografia",
        r"ecografia\s+mamaria\s+y\s+(de\s+la\s+region\s+)?axilar",
        r"sugerimos\s+ecografia\s+mamaria",
        r"ecografia\s+mamaria\s+bilateral",
        r"compresion\s+focalizada",
        r"incidencias\s+con\s+(magnificacion|compresion)",
        r"complementar\s+con\s+(ecografia|rm)",
        r"\brm\b|resonancia\s+magnetica",
        r"magnificacion(es)?",
    ],
    "correlacion_ecografica": [
        r"correlacion\s+(con\s+)?ecograf",
        r"correlacionar\s+(este\s+estudio\s+)?con\s+ecograf",
    ],
    "comparacion_estudios_previos": [
        r"comparacion\s+con\s+estudios?\s+(anteriores?|previos?)",
        r"correlacion\s+con\s+estudios?\s+(anteriores?|previos?)",
        r"comparacion\s+con\s+(mamografia|estudios?)\s+(anteriores?|previos?)",
        r"correlacion\s+con\s+los\s+mismos",
        r"para\s+apreciar\s+evolucion",
    ],
    "control_anual": [
        r"control\s+mamografico\s+anual",
        r"control\s+anual",
        r"mamografia\s+de\s+control\s+anual",
        r"controles?\s+anuales?",
        r"control\s+mamografico\s+y\s+ecografico\s+anual",
        r"control\s+ecografico\s+y\s+mamografico\s+anual",
        r"controles?\s+mamografico[s]?\s+y\s+ecografico[s]?\s+anuales?",
        r"controles?\s+ecografico[s]?\s+y\s+mamografico[s]?\s+anuales?",
    ],
    "control_corto_plazo": [
        r"control\s+semestral",
        r"control\s+en\s+6\s+meses",
        r"control\s+en\s+seis\s+meses",
        r"control\s+a\s+los?\s+6\s+meses",
        r"seguimiento\s+a\s+6\s+meses",
        r"control\s+en\s+3\s+meses",
        r"control\s+ecografico\s+semestral",
        r"control\s+ecografico\s+en\s+(6|seis)\s+meses",
        r"control\s+ecografico\s+y\s+mamografia.{0,30}en\s+(6|seis)\s+meses",
    ],
    "biopsia_histologia": [
        r"caracterizacion\s+histologica",
        r"estudio\s+histologico",
        r"biopsia",
        r"muestra\s+histopatologica",
    ],
    "criterio_medico": [
        r"criterio\s+del\s+medico\s+tratante",
        r"segun\s+criterio\s+medico",
        r"decidir\s+conducta",
        r"controles\s+habituales",
        r"control\s+mamografico\s+bianual",
        r"control\s+bianual",
        r"\bbianual(es)?\b",
        r"controles?\s+a\s+corto\s+plazo",
        r"antecedentes\s+clinicos.{0,50}patologia\s+extramamaria",
        r"descartar\s+patologia\s+extramamaria",
    ],
    "derivacion_oncologica": [
        r"derivacion\s+a\s+(oncolog|especialista)",
        r"manejo\s+oncologico",
        r"evaluacion\s+oncologica",
    ],
}

n_patrones = sum(len(p) for p in PATRONES_POR_CATEGORIA.values())
print(f"Categorías definidas: {len(PATRONES_POR_CATEGORIA)}")
print(f"Patrones regex totales: {n_patrones}")


Categorías definidas: 8
Patrones regex totales: 56


---

## Paso 7 — Aplicación del clasificador al corpus completo


In [6]:
def detectar_categorias(texto_normalizado):
    """Devuelve la lista de categorías detectadas en un texto ya normalizado."""
    encontradas = []
    for categoria, patrones in PATRONES_POR_CATEGORIA.items():
        patron_combinado = "|".join(patrones)
        if re.search(patron_combinado, texto_normalizado):
            encontradas.append(categoria)
    return encontradas

def categoria_principal(categorias_detectadas):
    """Devuelve la categoría de mayor prioridad según la jerarquía clínica."""
    if not categorias_detectadas:
        return None
    for cat in JERARQUIA_CLINICA:
        if cat in categorias_detectadas:
            return cat
    return categorias_detectadas[0]

# Aplicar al corpus completo
df_rec["rec_normalizada"] = df_rec["Recommendations"].apply(normalizar_texto)
df_rec["categorias"] = df_rec["rec_normalizada"].apply(detectar_categorias)
df_rec["categoria_principal"] = df_rec["categorias"].apply(categoria_principal)
df_rec["sin_categoria"] = df_rec["categorias"].apply(lambda x: len(x) == 0)

n_sin = df_rec["sin_categoria"].sum()
print("=" * 70)
print("COBERTURA DEL CLASIFICADOR REGEX")
print("=" * 70)
print(f"Total con recomendación: {len(df_rec)}")
print(f"Clasificados: {len(df_rec) - n_sin} ({100*(len(df_rec) - n_sin)/len(df_rec):.2f}%)")
print(f"Sin clasificar: {n_sin} ({100*n_sin/len(df_rec):.2f}%)")


COBERTURA DEL CLASIFICADOR REGEX
Total con recomendación: 4347
Clasificados: 4347 (100.00%)
Sin clasificar: 0 (0.00%)


### 7.1 Distribución de categorías detectadas


In [7]:
from collections import Counter
todas_cats = []
for lista in df_rec["categorias"]:
    todas_cats.extend(lista)
contador = Counter(todas_cats)

print("DISTRIBUCION DE CATEGORIAS DETECTADAS (informes pueden tener más de una)")
print("=" * 70)
for cat, n in contador.most_common():
    pct = 100 * n / len(df_rec)
    print(f"  {cat:35s}: {n:5d} ({pct:5.1f}%)")

print("\nDistribución del número de categorías por informe:")
print(df_rec["categorias"].apply(len).value_counts().sort_index())


DISTRIBUCION DE CATEGORIAS DETECTADAS (informes pueden tener más de una)
  estudio_complementario_imagen      :  2346 ( 54.0%)
  control_anual                      :  2019 ( 46.4%)
  correlacion_ecografica             :  2007 ( 46.2%)
  criterio_medico                    :   396 (  9.1%)
  biopsia_histologia                 :    58 (  1.3%)
  control_corto_plazo                :    47 (  1.1%)
  comparacion_estudios_previos       :    23 (  0.5%)

Distribución del número de categorías por informe:
categorias
1    2300
2    1545
3     502
Name: count, dtype: int64


### 7.2 Distribución de la categoría principal por BI-RADS


In [8]:
matriz = pd.crosstab(
    df_rec["BI-RADS"],
    df_rec["categoria_principal"],
    margins=True,
    margins_name="Total"
)
print("CATEGORIA PRINCIPAL POR BI-RADS")
print("=" * 70)
print(matriz)


CATEGORIA PRINCIPAL POR BI-RADS
categoria_principal  biopsia_histologia  comparacion_estudios_previos  \
BI-RADS                                                                 
0                                     8                            13   
1                                     0                             0   
2                                     1                             4   
3                                     0                             0   
4                                    35                             0   
5                                    14                             0   
Total                                58                            17   

categoria_principal  control_anual  control_corto_plazo  \
BI-RADS                                                   
0                                0                   10   
1                              211                    0   
2                             1070                    0   
3              

---

## Paso 8 — Evolución de la cobertura a lo largo del proceso

Para documentar metodológicamente, registro cómo evolucionó la cobertura mientras refiné los patrones regex y el normalizador:

| Iteración | Casos sin clasificar | Cobertura | Cambios |
|---|---|---|---|
| V1 — regex base sin normalizar | 190 / 4 347 | 95.63% | Patrones iniciales por categoría |
| V2 — con normalización + patrones ampliados | 41 / 4 347 | 99.06% | NFKD + diccionario base de typos + más variantes |
| V3 — fixes adicionales | 20 / 4 347 | 99.54% | Más typos + control bianual + 'controles habituales' |
| **V4 — versión final** | **0 / 4 347** | **100.00%** | Órdenes invertidos, palabras intermedias, último typo (CONTRO) |

**Lectura crítica**: aunque alcanzamos 100% en este corpus, la cobertura sobre otros corpus (otros centros asistenciales, distintos países) será menor. Por eso el módulo final incluirá una **capa de similitud TF-IDF** como red de seguridad para vocabulario no contemplado.


---

## Paso 9 — Limitaciones reconocidas y diseño del fallback TF-IDF

### Limitación principal: sobreajuste a este corpus

El 100% de cobertura aplica sobre el corpus de Vázquez Noguera et al. (2025). Estimaciones de pérdida de cobertura en otros escenarios:

| Escenario | Pérdida estimada |
|---|---|
| Otro centro chileno con redacción similar | 5-20% |
| Centro en otro país hispanohablante | 20-40% |
| Informes con redacción muy distinta a las plantillas | 40-60% |

### Estrategia de mitigación: arquitectura en capas

El módulo `src/extractor_recomendacion.py` implementará:

```
[texto crudo]
     ↓
Capa 1: Normalizador ortográfico (NFKD + diccionario typos)
     ↓
Capa 2: Regex sobre vocabulario clínico (100% en corpus actual)
     ↓
¿categoría detectada?
  ├─ SÍ → devolver, confianza=alta
  └─ NO → ir a Capa 3
     ↓
Capa 3: Similitud TF-IDF contra frases de referencia (umbral 0.55)
     ↓
¿similitud alta?
  ├─ SÍ → devolver, confianza=media + nota
  └─ NO → categoría='ambigua' + alerta para revisión humana
```

### Por qué TF-IDF y no DistilBETO

DistilBETO daría mejor robustez semántica, pero requeriría:
- Dataset etiquetado de recomendaciones (no existe)
- Fine-tuning adicional
- Cómputo en inferencia

TF-IDF es suficiente para el MVP porque:
- No requiere etiquetado adicional
- Es determinista y auditable
- Captura sinónimos y reordenamientos a nivel léxico


---

## Paso 10 — Guardar resultados del análisis


In [9]:
import os
import json

os.makedirs("./anexos", exist_ok=True)

# Guardar clasificación de recomendaciones
df_rec[["BI-RADS", "Recommendations", "rec_normalizada", "categorias", "categoria_principal"]].to_csv(
    "./anexos/clasificacion_recomendaciones.csv", index=False
)
print("OK Guardado: notebooks/anexos/clasificacion_recomendaciones.csv")

# Resumen de la exploración
resumen = {
    "total_con_recomendacion": int(len(df_rec)),
    "total_sin_recomendacion": int(len(df) - len(df_rec)),
    "cobertura_regex_pct": 100.00,
    "casos_sin_clasificar": int(df_rec["sin_categoria"].sum()),
    "categorias_definidas": list(PATRONES_POR_CATEGORIA.keys()),
    "jerarquia_clinica": JERARQUIA_CLINICA,
    "typos_corregidos": list(TYPOS_CLINICOS.values()),
    "distribucion_categorias": {cat: int(n) for cat, n in contador.most_common()},
    "distribucion_num_categorias": df_rec["categorias"].apply(len).value_counts().sort_index().to_dict(),
}

with open("./anexos/resumen_exploracion_recomendaciones.json", "w", encoding="utf-8") as f:
    json.dump(resumen, f, indent=2, ensure_ascii=False, default=str)

print("OK Guardado: notebooks/anexos/resumen_exploracion_recomendaciones.json")
print("\nResumen:")
for k, v in resumen.items():
    if isinstance(v, dict):
        print(f"  {k}:")
        for sk, sv in v.items():
            print(f"    {sk}: {sv}")
    elif isinstance(v, list):
        print(f"  {k}: {len(v)} elementos")
    else:
        print(f"  {k}: {v}")


OK Guardado: notebooks/anexos/clasificacion_recomendaciones.csv
OK Guardado: notebooks/anexos/resumen_exploracion_recomendaciones.json

Resumen:
  total_con_recomendacion: 4347
  total_sin_recomendacion: 10
  cobertura_regex_pct: 100.0
  casos_sin_clasificar: 0
  categorias_definidas: 8 elementos
  jerarquia_clinica: 8 elementos
  typos_corregidos: 9 elementos
  distribucion_categorias:
    estudio_complementario_imagen: 2346
    control_anual: 2019
    correlacion_ecografica: 2007
    criterio_medico: 396
    biopsia_histologia: 58
    control_corto_plazo: 47
    comparacion_estudios_previos: 23
  distribucion_num_categorias:
    1: 2300
    2: 1545
    3: 502


---

## Conclusiones del análisis exploratorio

### Hallazgos principales

1. **Cobertura regex de 100% sobre este corpus** tras refinamientos iterativos
2. **8 categorías clínicas** capturan la totalidad de las recomendaciones del dataset
3. **9 typos del corpus** fueron identificados y corregidos en el normalizador
4. **El 47% de los informes combina 2 o más categorías** (decisión: devolver lista, elegir principal por jerarquía)
5. **La jerarquía clínica prioriza dudas diagnósticas** sobre planes con plazo definido (principio del peor caso)

### Decisiones de diseño documentadas

- `controles habituales` → `criterio_medico` (es vago, requiere supervisión)
- `control bianual` → `criterio_medico` (no es estándar ACR, caso particular)
- `control manual` → typo de `anual` (autocorrector)
- `control mamográfico y ecográfico anual` → `control_anual` (ambos estudios a 12 meses)
- `controles a corto plazo` (sin tiempo) → `criterio_medico` (vago)
- `descartar patología extramamaria` → `criterio_medico` (delegación al tratante)

### Limitación honesta para el informe final

Las regex están optimizadas para este corpus. La generalización a otros centros requiere mantenimiento del vocabulario clínico. El módulo final mitiga esto con una capa de similitud TF-IDF que actúa como red de seguridad cuando las regex no encuentran coincidencia.

### Siguiente paso

Construir `src/extractor_recomendacion.py` con dos funciones públicas:
- `extraer_texto_recomendacion(recommendations_col, full_report=None)` → devuelve el texto + metadatos
- `clasificar_recomendacion(texto_normalizado)` → clasifica con regex (capa 2) o TF-IDF (capa 3 fallback)


In [1]:
import pandas as pd

# Cargar las clasificaciones que el nb 06 ya guardó
df_clasif = pd.read_csv("../notebooks/anexos/clasificacion_recomendaciones.csv")

# La columna 'categorias' viene como string '[...]'; convertir a lista
import ast
df_clasif["categorias"] = df_clasif["categorias"].apply(ast.literal_eval)

# Tabla normativa ACR
TABLA_ACR = {
    0: ("estudio_complementario_imagen", "alta"),
    1: ("control_anual", "baja"),
    2: ("control_anual", "baja"),
    3: ("control_corto_plazo", "media"),
    4: ("biopsia_histologia", "alta"),
    5: ("biopsia_histologia", "critica"),
    6: ("derivacion_oncologica", "media"),
}

JERARQUIA = [
    "biopsia_histologia",
    "derivacion_oncologica",
    "estudio_complementario_imagen",
    "correlacion_ecografica",
    "comparacion_estudios_previos",
    "control_corto_plazo",
    "control_anual",
    "criterio_medico",
]

def top_2_por_jerarquia(categorias):
    """Devuelve las top 2 categorías según la jerarquía clínica."""
    if not categorias:
        return []
    unicas = set(categorias)
    ordenadas = [c for c in JERARQUIA if c in unicas]
    return ordenadas[:2]

def es_coherente(birads, categorias_detectadas):
    """Coteja BI-RADS vs categorías detectadas según norma ACR (top 2)."""
    if birads not in TABLA_ACR:
        return None, None, None
    esperada, severidad = TABLA_ACR[birads]
    top2 = top_2_por_jerarquia(categorias_detectadas)
    coherente = esperada in top2
    return coherente, esperada, severidad

# Aplicar al corpus
df_clasif["top2"] = df_clasif["categorias"].apply(top_2_por_jerarquia)
df_clasif[["coherente", "esperada_acr", "severidad"]] = df_clasif.apply(
    lambda row: pd.Series(es_coherente(row["BI-RADS"], row["categorias"])),
    axis=1
)

print("=" * 75)
print("PRE-VALIDACIÓN: COTEJO ACR APLICADO AL CORPUS COMPLETO")
print("=" * 75)
print(f"\nTotal informes con recomendación: {len(df_clasif)}")
print(f"  Coherentes:    {df_clasif['coherente'].sum()} ({100*df_clasif['coherente'].sum()/len(df_clasif):.1f}%)")
print(f"  Incoherentes:  {(~df_clasif['coherente']).sum()} ({100*(~df_clasif['coherente']).sum()/len(df_clasif):.1f}%)")

# Alertas por severidad
print("\n--- ALERTAS POR SEVERIDAD ---")
alertas = df_clasif[~df_clasif["coherente"]]
print(alertas["severidad"].value_counts())

# Alertas por BI-RADS
print("\n--- ALERTAS POR BI-RADS ---")
print(pd.crosstab(alertas["BI-RADS"], alertas["severidad"], margins=True))

# Mostrar alertas críticas y altas (que son las clínicamente importantes)
print("\n--- ALERTAS CRÍTICAS Y ALTAS (clínicamente relevantes) ---")
graves = alertas[alertas["severidad"].isin(["critica", "alta"])].copy()
print(f"Total: {len(graves)}")
print("\nDesglose por BI-RADS:")
for birads, grupo in graves.groupby("BI-RADS"):
    print(f"\nBI-RADS {birads} ({len(grupo)} alertas):")
    print(f"  ACR esperaba: {TABLA_ACR.get(birads, ('?','?'))[0]}")
    print(f"  Categorías top 2 más frecuentes:")
    top_cats = grupo["top2"].apply(tuple).value_counts().head(3)
    for combo, n in top_cats.items():
        print(f"    {list(combo)}: {n} informes")

PRE-VALIDACIÓN: COTEJO ACR APLICADO AL CORPUS COMPLETO

Total informes con recomendación: 4347
  Coherentes:    2672 (61.5%)
  Incoherentes:  1675 (38.5%)

--- ALERTAS POR SEVERIDAD ---
severidad
baja       1555
alta         68
media        50
critica       2
Name: count, dtype: int64

--- ALERTAS POR BI-RADS ---
severidad  alta  baja  critica  media   All
BI-RADS                                    
0            51     0        0      0    51
1             0   332        0      0   332
2             0  1223        0      0  1223
3             0     0        0     50    50
4            17     0        0      0    17
5             0     0        2      0     2
All          68  1555        2     50  1675

--- ALERTAS CRÍTICAS Y ALTAS (clínicamente relevantes) ---
Total: 70

Desglose por BI-RADS:

BI-RADS 0 (51 alertas):
  ACR esperaba: estudio_complementario_imagen
  Categorías top 2 más frecuentes:
    ['correlacion_ecografica']: 15 informes
    ['comparacion_estudios_previos']: 13 infor

In [2]:
# Diagnosticar los BI-RADS 2 incoherentes
b2_incoherente = df_clasif[
    (df_clasif["BI-RADS"] == 2) &
    (~df_clasif["coherente"])
].copy()

print(f"BI-RADS 2 incoherentes: {len(b2_incoherente)}")
print(f"\nTop 5 combinaciones de 'top2' en estos casos:")
print(b2_incoherente["top2"].apply(tuple).value_counts().head(5))

print(f"\nTop 5 combinaciones de TODAS las categorías detectadas:")
print(b2_incoherente["categorias"].apply(tuple).value_counts().head(5))

print(f"\n¿Cuántos tienen 'control_anual' EN LA LISTA pero no en top 2?")
tiene_control_no_top2 = b2_incoherente[
    b2_incoherente["categorias"].apply(lambda x: "control_anual" in x)
]
print(f"  {len(tiene_control_no_top2)} ({100*len(tiene_control_no_top2)/len(b2_incoherente):.1f}%)")

# Ver ejemplos de los que tienen control_anual pero no fueron clasificados como coherentes
print(f"\nEjemplos:")
for idx, row in tiene_control_no_top2.head(5).iterrows():
    print(f"\n[idx {idx}]")
    print(f"  Categorías: {row['categorias']}")
    print(f"  Top 2: {row['top2']}")
    print(f"  ACR esperaba: control_anual")
    print(f"  Texto original: {row['Recommendations'][:150]}")

BI-RADS 2 incoherentes: 1223

Top 5 combinaciones de 'top2' en estos casos:
top2
(estudio_complementario_imagen, correlacion_ecografica)    995
(estudio_complementario_imagen,)                            85
(criterio_medico,)                                          68
(correlacion_ecografica, criterio_medico)                   47
(correlacion_ecografica,)                                   19
Name: count, dtype: int64

Top 5 combinaciones de TODAS las categorías detectadas:
categorias
(estudio_complementario_imagen, correlacion_ecografica)                     611
(estudio_complementario_imagen, correlacion_ecografica, control_anual)      312
(estudio_complementario_imagen,)                                             85
(estudio_complementario_imagen, correlacion_ecografica, criterio_medico)     72
(criterio_medico,)                                                           68
Name: count, dtype: int64

¿Cuántos tienen 'control_anual' EN LA LISTA pero no en top 2?
  312 (25.5%)

Ejempl

In [3]:
def es_mas_urgente(cat_a, cat_b):
    """Devuelve True si cat_a es más urgente que cat_b según jerarquía."""
    try:
        return JERARQUIA.index(cat_a) < JERARQUIA.index(cat_b)
    except ValueError:
        return False

def cotejar_nueva(birads, categorias):
    """Nueva lógica de cotejo en dos pasos."""
    if birads not in TABLA_ACR:
        return None, None, None
    esperada, severidad = TABLA_ACR[birads]
    if not categorias:
        return False, esperada, severidad

    # Paso 1: ¿esperada está en la lista?
    if esperada in categorias:
        return "coherente", esperada, severidad

    # Paso 2: ¿principal detectada es más urgente?
    unicas = set(categorias)
    principal = next((c for c in JERARQUIA if c in unicas), None)
    if principal and es_mas_urgente(principal, esperada):
        return "coherente_con_precaucion", esperada, severidad

    # No cumple: alerta real
    return "incoherente", esperada, severidad


# Aplicar al corpus
df_clasif[["estado_cotejo", "esperada_acr", "severidad"]] = df_clasif.apply(
    lambda row: pd.Series(cotejar_nueva(row["BI-RADS"], row["categorias"])),
    axis=1
)

print("=" * 75)
print("NUEVA LÓGICA DE COTEJO (dos pasos)")
print("=" * 75)
print(f"\nDistribución de estados:")
print(df_clasif["estado_cotejo"].value_counts())
print(f"\nPorcentajes:")
for estado, n in df_clasif["estado_cotejo"].value_counts().items():
    print(f"  {estado:30s}: {n:5d} ({100*n/len(df_clasif):5.1f}%)")

# Alertas REALES (incoherente) por severidad
print("\n--- ALERTAS REALES POR SEVERIDAD ---")
alertas_reales = df_clasif[df_clasif["estado_cotejo"] == "incoherente"]
print(alertas_reales["severidad"].value_counts())

# Alertas REALES por BI-RADS
print("\n--- ALERTAS REALES POR BI-RADS ---")
print(pd.crosstab(alertas_reales["BI-RADS"], alertas_reales["severidad"], margins=True))

# Mostrar todas las críticas y altas (las clínicamente importantes)
print("\n--- ALERTAS CRÍTICAS Y ALTAS ---")
graves = alertas_reales[alertas_reales["severidad"].isin(["critica", "alta"])]
print(f"Total: {len(graves)}")
print(f"\nEjemplos:")
for idx, row in graves.head(8).iterrows():
    print(f"\n[idx {idx}] BI-RADS {row['BI-RADS']} | severidad: {row['severidad']}")
    print(f"  Esperado ACR: {row['esperada_acr']}")
    print(f"  Detectado: {row['categorias']}")
    print(f"  Texto: {row['Recommendations'][:150]}")

NUEVA LÓGICA DE COTEJO (dos pasos)

Distribución de estados:
estado_cotejo
coherente                   3010
coherente_con_precaucion    1166
incoherente                  171
Name: count, dtype: int64

Porcentajes:
  coherente                     :  3010 ( 69.2%)
  coherente_con_precaucion      :  1166 ( 26.8%)
  incoherente                   :   171 (  3.9%)

--- ALERTAS REALES POR SEVERIDAD ---
severidad
baja       99
alta       62
media       8
critica     2
Name: count, dtype: int64

--- ALERTAS REALES POR BI-RADS ---
severidad  alta  baja  critica  media  All
BI-RADS                                   
0            45     0        0      0   45
1             0    31        0      0   31
2             0    68        0      0   68
3             0     0        0      8    8
4            17     0        0      0   17
5             0     0        2      0    2
All          62    99        2      8  171

--- ALERTAS CRÍTICAS Y ALTAS ---
Total: 64

Ejemplos:

[idx 0] BI-RADS 0 | severidad:

In [4]:
# Verificar las 51 alertas altas de BI-RADS 0
print("=" * 75)
print("VERIFICACIÓN: BI-RADS 0 alertas altas - ¿bug del extractor o alerta real?")
print("=" * 75)

birads0_alertas = df_clasif[
    (df_clasif["BI-RADS"] == 0) &
    (df_clasif["estado_cotejo"] == "incoherente")
].copy()

print(f"\nTotal alertas BI-RADS 0: {len(birads0_alertas)}")
print(f"\nDistribución de categorías detectadas:")
print(birads0_alertas["categorias"].apply(tuple).value_counts())

print(f"\nRevisión textual: ¿el texto MENCIONA 'ecografia' o 'imagen complementaria' pero el extractor no la capturó?")
import re
import unicodedata

def normalizar(t):
    t = unicodedata.normalize("NFKD", t)
    t = "".join(c for c in t if not unicodedata.combining(c))
    return t.lower()

# Contar cuántos casos contienen "ecografia" en el texto pero NO está en las categorías
con_eco_en_texto = 0
sin_eco_en_texto = 0

for idx, row in birads0_alertas.iterrows():
    texto_norm = normalizar(str(row["Recommendations"]))
    tiene_eco_o_imagen = bool(re.search(r"ecograf|magnificac|compresion|resonanc|\brm\b", texto_norm))
    en_categorias = "estudio_complementario_imagen" in row["categorias"]
    
    if tiene_eco_o_imagen and not en_categorias:
        con_eco_en_texto += 1
    elif not tiene_eco_o_imagen:
        sin_eco_en_texto += 1

print(f"\n  Casos donde texto MENCIONA estudio complementario pero extractor no lo detectó: {con_eco_en_texto}")
print(f"  Casos donde texto NO menciona estudio complementario (alerta real): {sin_eco_en_texto}")

# Mostrar ejemplos de cada caso
print(f"\n--- EJEMPLOS: texto SÍ menciona eco pero no fue detectado (bug del extractor) ---")
for idx, row in birads0_alertas.iterrows():
    texto_norm = normalizar(str(row["Recommendations"]))
    tiene_eco = bool(re.search(r"ecograf|magnificac|compresion|resonanc|\brm\b", texto_norm))
    en_cat = "estudio_complementario_imagen" in row["categorias"]
    if tiene_eco and not en_cat:
        print(f"\n[idx {idx}] cats: {row['categorias']}")
        print(f"  Texto: {row['Recommendations'][:180]}")
        if list(birads0_alertas[~birads0_alertas["categorias"].apply(lambda x: "estudio_complementario_imagen" in x)].index).count(idx) >= 3:
            break

print(f"\n--- EJEMPLOS: texto NO menciona eco (alerta REAL) ---")
contador = 0
for idx, row in birads0_alertas.iterrows():
    texto_norm = normalizar(str(row["Recommendations"]))
    tiene_eco = bool(re.search(r"ecograf|magnificac|compresion|resonanc|\brm\b", texto_norm))
    if not tiene_eco:
        contador += 1
        print(f"\n[idx {idx}] cats: {row['categorias']}")
        print(f"  Texto: {row['Recommendations'][:180]}")
        if contador >= 3:
            break

VERIFICACIÓN: BI-RADS 0 alertas altas - ¿bug del extractor o alerta real?

Total alertas BI-RADS 0: 45

Distribución de categorías detectadas:
categorias
(correlacion_ecografica,)                 15
(comparacion_estudios_previos,)           13
(criterio_medico,)                         7
(control_corto_plazo, criterio_medico)     7
(control_corto_plazo,)                     3
Name: count, dtype: int64

Revisión textual: ¿el texto MENCIONA 'ecografia' o 'imagen complementaria' pero el extractor no la capturó?

  Casos donde texto MENCIONA estudio complementario pero extractor no lo detectó: 37
  Casos donde texto NO menciona estudio complementario (alerta real): 8

--- EJEMPLOS: texto SÍ menciona eco pero no fue detectado (bug del extractor) ---

[idx 0] cats: ['comparacion_estudios_previos']
  Texto: - SE SUGIERE ECOGRAFÍA MAMARIA Y CORRELACIÓN CON ESTUDIOS ANTERIORES PARA POSTERIOR RECATEGORIZACIÓN.

[idx 2] cats: ['correlacion_ecografica']
  Texto: - SE SUGIERE CORRELACIÓN CON ECOGRA

In [5]:
# Diagnóstico: ¿cuántos casos contienen "ecografia mamaria ... para ... recategorizacion"
# pero no fueron capturados por la regex actual?

import re
import unicodedata

def normalizar(t):
    t = unicodedata.normalize("NFKD", t)
    t = "".join(c for c in t if not unicodedata.combining(c))
    return t.lower()

# Patrón AMPLIADO propuesto: "ecografia mamaria" + cualquier cosa + "para recategorizacion"
patron_propuesto = re.compile(r"ecografia\s+mamaria\s+.{0,80}?para\s+(posterior\s+)?recategorizacion")

# Buscar en TODO el corpus (no solo en los incoherentes)
df_clasif["rec_norm"] = df_clasif["Recommendations"].astype(str).apply(normalizar)

# Casos donde el patrón nuevo matchearía pero estudio_complementario_imagen NO está en categorías
df_clasif["nuevo_patron_matchea"] = df_clasif["rec_norm"].apply(
    lambda x: bool(patron_propuesto.search(x))
)
df_clasif["tiene_estudio_complementario"] = df_clasif["categorias"].apply(
    lambda x: "estudio_complementario_imagen" in x
)

# Casos que recuperaríamos con el patrón nuevo
recuperables = df_clasif[
    df_clasif["nuevo_patron_matchea"] &
    ~df_clasif["tiene_estudio_complementario"]
]
print(f"Casos recuperables con patrón ampliado: {len(recuperables)}")
print(f"\nDesglose por BI-RADS:")
print(recuperables["BI-RADS"].value_counts().sort_index())

print(f"\nEjemplos:")
for idx, row in recuperables.head(5).iterrows():
    print(f"\n[idx {idx}] BI-RADS {row['BI-RADS']}")
    print(f"  Cats actuales: {row['categorias']}")
    print(f"  Texto: {row['Recommendations'][:200]}")

Casos recuperables con patrón ampliado: 13

Desglose por BI-RADS:
BI-RADS
0    13
Name: count, dtype: int64

Ejemplos:

[idx 0] BI-RADS 0
  Cats actuales: ['comparacion_estudios_previos']
  Texto: - SE SUGIERE ECOGRAFÍA MAMARIA Y CORRELACIÓN CON ESTUDIOS ANTERIORES PARA POSTERIOR RECATEGORIZACIÓN.

[idx 42] BI-RADS 0
  Cats actuales: ['correlacion_ecografica']
  Texto: - SE SUGIERE CORRELACIÓN CON ECOGRAFÍA MAMARIA Y CON ESTUDIOS ANTERIORES PARA POSTERIOR RECATEGORIZACIÓN.-

[idx 252] BI-RADS 0
  Cats actuales: ['comparacion_estudios_previos']
  Texto: - SE SUGIERE ECOGRAFÍA MAMARIA O CORRELACIÓN CON ESTUDIOS ANTERIORES PARA POSTERIOR RECATEGORIZACIÓN.

[idx 299] BI-RADS 0
  Cats actuales: ['comparacion_estudios_previos']
  Texto: - SE SUGIERE ECOGRAFÍA MAMARIA Y COMPARACIÓN CON ESTUDIOS PREVIOS PARA POSTERIOR RECATEGORIZACIÓN.-

[idx 332] BI-RADS 0
  Cats actuales: ['comparacion_estudios_previos']
  Texto: - SE SUGIERE ECOGRAFÍA MAMARIA Y CORRELACIÓN CON ESTUDIOS ANTERIORES PARA POSTE

In [6]:
import re
import unicodedata

def normalizar(t):
    t = unicodedata.normalize("NFKD", t)
    t = "".join(c for c in t if not unicodedata.combining(c))
    return t.lower()

# Patrón propuesto del fix
patron_propuesto = re.compile(r"ecografia\s+mamaria\s+.{0,80}?para\s+(posterior\s+)?recategorizacion")

df_clasif["rec_norm"] = df_clasif["Recommendations"].astype(str).apply(normalizar)
df_clasif["menciona_eco"] = df_clasif["rec_norm"].str.contains(
    r"ecograf|magnificac|compresion|resonanc|\brm\b", regex=True, na=False
)
df_clasif["tiene_estudio_complementario"] = df_clasif["categorias"].apply(
    lambda x: "estudio_complementario_imagen" in x
)
df_clasif["nuevo_patron_matchea"] = df_clasif["rec_norm"].apply(
    lambda x: bool(patron_propuesto.search(x))
)

# BI-RADS 0 alertados con la nueva lógica de cotejo
b0_alertas = df_clasif[
    (df_clasif["BI-RADS"] == 0) &
    (df_clasif["estado_cotejo"] == "incoherente")
].copy()

# Los 24 "casos intermedios": menciona eco, NO tiene estudio_complementario, fix NO los recupera
casos_intermedios = b0_alertas[
    b0_alertas["menciona_eco"] &
    ~b0_alertas["tiene_estudio_complementario"] &
    ~b0_alertas["nuevo_patron_matchea"]
]

print("=" * 75)
print(f"ANÁLISIS DETALLADO: {len(casos_intermedios)} CASOS INTERMEDIOS")
print("=" * 75)
print("\nDistribución de categorías detectadas:")
print(casos_intermedios["categorias"].apply(tuple).value_counts())

print(f"\n--- TODOS LOS {len(casos_intermedios)} CASOS ---")
for i, (idx, row) in enumerate(casos_intermedios.iterrows(), 1):
    print(f"\n[{i}/{len(casos_intermedios)}] idx {idx}")
    print(f"  Cats detectadas: {row['categorias']}")
    print(f"  Texto: {row['Recommendations'][:220]}")

ANÁLISIS DETALLADO: 24 CASOS INTERMEDIOS

Distribución de categorías detectadas:
categorias
(correlacion_ecografica,)                 8
(control_corto_plazo, criterio_medico)    7
(comparacion_estudios_previos,)           5
(criterio_medico,)                        2
(control_corto_plazo,)                    2
Name: count, dtype: int64

--- TODOS LOS 24 CASOS ---

[1/24] idx 2
  Cats detectadas: ['correlacion_ecografica']
  Texto: - SE SUGIERE CORRELACIÓN CON ECOGRAFÍA MAMARIA E INCIDENCIAS MAGNIFICADAS PARA SU POSTERIOR RECATEGORIZACIÓN.-

[2/24] idx 5
  Cats detectadas: ['correlacion_ecografica']
  Texto: - SE SUGIERE CORRELACIÓN CON ECOGRAFÍA MAMARIA Y CON ESTUDIOS ANTERIORES.-

[3/24] idx 17
  Cats detectadas: ['criterio_medico']
  Texto: - SE SUGIERE ECOGRAFÍA MAMARIA EN PRIMER TÉRMINO Y OTROS ESTUDIOS SEGÚN CRITERIO DEL MÉDICO TRATANTE.

[4/24] idx 52
  Cats detectadas: ['correlacion_ecografica']
  Texto: - SE SUGIERE CORRELACIÓN CON ECOGRAFÍA MAMARIA.

[5/24] idx 95
  Cats detec

In [7]:
b_cortos = casos_intermedios[
    casos_intermedios["categorias"].apply(lambda x: "control_corto_plazo" in x)
]
print(f"--- {len(b_cortos)} casos con control_corto_plazo ---")
for i, (idx, row) in enumerate(b_cortos.iterrows(), 1):
    print(f"\n[{i}] idx {idx} | cats: {row['categorias']}")
    print(f"  Texto: {row['Recommendations'][:230]}")

--- 9 casos con control_corto_plazo ---

[1] idx 1543 | cats: ['control_corto_plazo', 'criterio_medico']
  Texto: - SE SUGIERE CONTROL ECOGRÁFICO EN SEIS MESES Y CONTROL MAMOGRÁFICO SEGÚN CRITERIO DEL MÉDICO TRATATNE

[2] idx 1934 | cats: ['control_corto_plazo', 'criterio_medico']
  Texto: - Se sugiere control ecográfico en seis meses y mamográfico según criterio del médico tratante.

[3] idx 2308 | cats: ['control_corto_plazo']
  Texto: - Se sugiere control ecográfico semestral.

[4] idx 2429 | cats: ['control_corto_plazo', 'criterio_medico']
  Texto: - Se sugiere control ecográfico semestral y mamográfico según criterio del médico tratante.

[5] idx 2837 | cats: ['control_corto_plazo', 'criterio_medico']
  Texto: - Se sugiere control ecográfico semestral y control mamográfico según criterio del médico tratante.

[6] idx 3188 | cats: ['control_corto_plazo']
  Texto: - Se sugiere control ecográfico y mamografía derecha en seis meses.

[7] idx 3257 | cats: ['control_corto_plazo', 'crite

In [8]:
b_criterio = casos_intermedios[
    casos_intermedios["categorias"].apply(lambda x: x == ["criterio_medico"])
]
print(f"\n--- {len(b_criterio)} casos solo con criterio_medico ---")
for i, (idx, row) in enumerate(b_criterio.iterrows(), 1):
    print(f"\n[{i}] idx {idx}")
    print(f"  Texto: {row['Recommendations'][:230]}")


--- 2 casos solo con criterio_medico ---

[1] idx 17
  Texto: - SE SUGIERE ECOGRAFÍA MAMARIA EN PRIMER TÉRMINO Y OTROS ESTUDIOS SEGÚN CRITERIO DEL MÉDICO TRATANTE.

[2] idx 2491
  Texto: - Se sugiere ecografía mamaria semestral y control mamográfico según criterio del médico tratante.


In [9]:
TABLA_ACR_FINAL = {
    0: {"esperada": "estudio_complementario_imagen",
        "equivalentes_aceptables": ["correlacion_ecografica", "comparacion_estudios_previos"],
        "equivalentes_con_notificacion": [],
        "severidad": "alta"},
    1: {"esperada": "control_anual",
        "equivalentes_aceptables": ["criterio_medico"],
        "equivalentes_con_notificacion": [],
        "severidad": "baja"},
    2: {"esperada": "control_anual",
        "equivalentes_aceptables": ["correlacion_ecografica", "criterio_medico"],
        "equivalentes_con_notificacion": ["control_corto_plazo"],
        "severidad": "baja"},
    3: {"esperada": "control_corto_plazo",
        "equivalentes_aceptables": ["biopsia_histologia"],
        "equivalentes_con_notificacion": [],
        "severidad": "media"},
    4: {"esperada": "biopsia_histologia",
        "equivalentes_aceptables": ["derivacion_oncologica"],
        "equivalentes_con_notificacion": [],
        "severidad": "alta"},
    5: {"esperada": "biopsia_histologia",
        "equivalentes_aceptables": [],
        "equivalentes_con_notificacion": [],
        "severidad": "critica"},
    6: {"esperada": "derivacion_oncologica",
        "equivalentes_aceptables": ["criterio_medico", "biopsia_histologia"],
        "equivalentes_con_notificacion": [],
        "severidad": "media"},
}

def cotejar_v4(birads, categorias):
    if birads not in TABLA_ACR_FINAL:
        return None, None, None
    cfg = TABLA_ACR_FINAL[birads]
    esperada = cfg["esperada"]
    equivalentes = cfg["equivalentes_aceptables"]
    notificacion = cfg["equivalentes_con_notificacion"]
    severidad = cfg["severidad"]

    if not categorias:
        return "incoherente", esperada, severidad

    # 1. esperada presente
    if esperada in categorias:
        return "coherente", esperada, severidad
    # 2. equivalente aceptable
    if any(e in categorias for e in equivalentes):
        return "coherente_equivalente", esperada, severidad
    # 3. equivalente con notificacion
    if any(e in categorias for e in notificacion):
        return "notificacion", esperada, "baja"
    # 4. principal mas urgente
    unicas = set(categorias)
    principal = next((c for c in JERARQUIA if c in unicas), None)
    if principal:
        try:
            if JERARQUIA.index(principal) < JERARQUIA.index(esperada):
                return "coherente_con_precaucion", esperada, severidad
        except ValueError:
            pass
    return "incoherente", esperada, severidad


df_clasif[["estado_v4", "esperada_v4", "severidad_v4"]] = df_clasif.apply(
    lambda row: pd.Series(cotejar_v4(row["BI-RADS"], row["categorias"])),
    axis=1
)

print("=" * 75)
print("LÓGICA DE COTEJO V4 (con correcciones clínicas)")
print("=" * 75)
print("\nDistribución de estados:")
distribucion = df_clasif["estado_v4"].value_counts()
print(distribucion)
print("\nPorcentajes:")
for estado, n in distribucion.items():
    print(f"  {estado:30s}: {n:5d} ({100*n/len(df_clasif):5.1f}%)")

# Alertas reales
print("\n" + "=" * 75)
print("ALERTAS REALES (estado=incoherente)")
print("=" * 75)
alertas = df_clasif[df_clasif["estado_v4"] == "incoherente"]
print(f"\nTotal alertas: {len(alertas)}")
print(f"\nPor severidad:")
print(alertas["severidad_v4"].value_counts())
print(f"\nPor BI-RADS y severidad:")
print(pd.crosstab(alertas["BI-RADS"], alertas["severidad_v4"], margins=True))

# Notificaciones (no alertas, pero info útil)
print("\n" + "=" * 75)
print("NOTIFICACIONES (alerta suave, no urgente)")
print("=" * 75)
notif = df_clasif[df_clasif["estado_v4"] == "notificacion"]
print(f"Total notificaciones: {len(notif)}")
print(f"\nPor BI-RADS:")
print(notif["BI-RADS"].value_counts())

# Alertas críticas y altas con ejemplos
print("\n" + "=" * 75)
print("ALERTAS CRÍTICAS Y ALTAS (clínicamente relevantes)")
print("=" * 75)
graves = alertas[alertas["severidad_v4"].isin(["critica", "alta"])]
print(f"Total críticas + altas: {len(graves)}")
for idx, row in graves.head(10).iterrows():
    print(f"\n[idx {idx}] BI-RADS {row['BI-RADS']} ({row['severidad_v4']})")
    print(f"  ACR esperaba: {row['esperada_v4']}")
    print(f"  Detectado:    {row['categorias']}")
    print(f"  Texto:        {row['Recommendations'][:170]}")

LÓGICA DE COTEJO V4 (con correcciones clínicas)

Distribución de estados:
estado_v4
coherente                   3010
coherente_equivalente        953
coherente_con_precaucion     340
incoherente                   44
Name: count, dtype: int64

Porcentajes:
  coherente                     :  3010 ( 69.2%)
  coherente_equivalente         :   953 ( 21.9%)
  coherente_con_precaucion      :   340 (  7.8%)
  incoherente                   :    44 (  1.0%)

ALERTAS REALES (estado=incoherente)

Total alertas: 44

Por severidad:
severidad_v4
alta       34
media       8
critica     2
Name: count, dtype: int64

Por BI-RADS y severidad:
severidad_v4  alta  critica  media  All
BI-RADS                                
0               17        0      0   17
3                0        0      8    8
4               17        0      0   17
5                0        2      0    2
All             34        2      8   44

NOTIFICACIONES (alerta suave, no urgente)
Total notificaciones: 0

Por BI-RADS:
Series([

In [10]:
# Ver TODAS las 44 alertas
print("=" * 75)
print("DESGLOSE COMPLETO DE LAS 44 ALERTAS REALES")
print("=" * 75)

print("\nPor severidad:")
print(alertas["severidad_v4"].value_counts())

print("\nPor BI-RADS:")
print(alertas["BI-RADS"].value_counts().sort_index())

print("\nCruce BI-RADS x severidad:")
print(pd.crosstab(alertas["BI-RADS"], alertas["severidad_v4"], margins=True))

# Ejemplos por cada combinación de BI-RADS x severidad
print("\n" + "=" * 75)
print("EJEMPLOS POR CADA TIPO DE ALERTA")
print("=" * 75)

for birads in sorted(alertas["BI-RADS"].unique()):
    sub = alertas[alertas["BI-RADS"] == birads]
    print(f"\n--- BI-RADS {birads} ({len(sub)} alertas) ---")
    for idx, row in sub.head(3).iterrows():
        print(f"\n  [idx {idx}] severidad: {row['severidad_v4']}")
        print(f"  Esperaba: {row['esperada_v4']}")
        print(f"  Detectado: {row['categorias']}")
        print(f"  Texto: {row['Recommendations'][:170]}")

DESGLOSE COMPLETO DE LAS 44 ALERTAS REALES

Por severidad:
severidad_v4
alta       34
media       8
critica     2
Name: count, dtype: int64

Por BI-RADS:
BI-RADS
0    17
3     8
4    17
5     2
Name: count, dtype: int64

Cruce BI-RADS x severidad:
severidad_v4  alta  critica  media  All
BI-RADS                                
0               17        0      0   17
3                0        0      8    8
4               17        0      0   17
5                0        2      0    2
All             34        2      8   44

EJEMPLOS POR CADA TIPO DE ALERTA

--- BI-RADS 0 (17 alertas) ---

  [idx 17] severidad: alta
  Esperaba: estudio_complementario_imagen
  Detectado: ['criterio_medico']
  Texto: - SE SUGIERE ECOGRAFÍA MAMARIA EN PRIMER TÉRMINO Y OTROS ESTUDIOS SEGÚN CRITERIO DEL MÉDICO TRATANTE.

  [idx 837] severidad: alta
  Esperaba: estudio_complementario_imagen
  Detectado: ['criterio_medico']
  Texto: - SE SUGIERE CONTROLES SEGÚN CRITERIO DEL MÉDICO TRATANTE.

  [idx 925] severid

In [11]:
print("=" * 75)
print("LOS 2 CASOS CRÍTICOS — BI-RADS 5 sin biopsia ni derivación oncológica")
print("=" * 75)

criticos = df_clasif[
    (df_clasif["BI-RADS"] == 5) &
    (df_clasif["estado_v4"] == "incoherente")
].copy()

for i, (idx, row) in enumerate(criticos.iterrows(), 1):
    print(f"\n{'─' * 75}")
    print(f"CASO CRÍTICO {i}/2 — Informe idx {idx}")
    print(f"{'─' * 75}")
    print(f"\n  BI-RADS declarado:         {row['BI-RADS']}")
    print(f"  Recomendación esperada ACR: {row['esperada_v4']}")
    print(f"  Severidad:                  {row['severidad_v4']}")
    print(f"\n  Categorías detectadas:")
    for cat in row['categorias']:
        print(f"    • {cat}")
    print(f"\n  TEXTO ORIGINAL DE LA RECOMENDACIÓN:")
    print(f"  ─────────────────────────────────────")
    print(f"  {row['Recommendations']}")
    
    # Si tenemos acceso al Full_Report para más contexto
    # (intentamos obtenerlo del dataset original)
    print()

# Buscar el contexto del Full_Report para entender el informe completo
print("\n" + "=" * 75)
print("CONTEXTO ADICIONAL: ¿qué dice el resto del informe?")
print("=" * 75)

# Cargar el corpus original para ver el Full_Report
df_original = pd.read_csv("../data/processed/reports_cleaned.csv")
for i, (idx, row) in enumerate(criticos.iterrows(), 1):
    full_report = df_original.loc[idx, "Full_Report"] if idx in df_original.index else None
    print(f"\n--- Informe crítico {i} (idx {idx}) ---")
    if full_report:
        # Mostrar últimos 500 caracteres del Full_Report (donde suele estar la conclusión)
        print(f"Final del Full_Report (últimos 600 chars):")
        print(f"{full_report[-600:]}")
    else:
        print("No se encontró el Full_Report en el dataset original")

LOS 2 CASOS CRÍTICOS — BI-RADS 5 sin biopsia ni derivación oncológica

───────────────────────────────────────────────────────────────────────────
CASO CRÍTICO 1/2 — Informe idx 3333
───────────────────────────────────────────────────────────────────────────

  BI-RADS declarado:         5
  Recomendación esperada ACR: biopsia_histologia
  Severidad:                  critica

  Categorías detectadas:
    • estudio_complementario_imagen
    • criterio_medico

  TEXTO ORIGINAL DE LA RECOMENDACIÓN:
  ─────────────────────────────────────
  - Se sugiere ecografía mamaria para posterior recategorización y decidir conducta.


───────────────────────────────────────────────────────────────────────────
CASO CRÍTICO 2/2 — Informe idx 4177
───────────────────────────────────────────────────────────────────────────

  BI-RADS declarado:         5
  Recomendación esperada ACR: biopsia_histologia
  Severidad:                  critica

  Categorías detectadas:
    • estudio_complementario_imagen

  

In [13]:
import sys
sys.path.insert(0, "..")  # agregar la raíz del proyecto al path
from src.extractor_birads import extraer_birads

# Cargar el corpus original
df_original = pd.read_csv("../data/processed/reports_cleaned.csv")

for idx in [3333, 4177]:
    full_report = df_original.loc[idx, "Full_Report"]
    birads_dataset = df_original.loc[idx, "BI-RADS"]
    resultado = extraer_birads(full_report)
    
    print(f"\n{'─' * 70}")
    print(f"Informe idx {idx}")
    print(f"{'─' * 70}")
    print(f"  BI-RADS en columna del dataset:  {birads_dataset}")
    print(f"  BI-RADS extraído por el sistema: {resultado['birads_conclusion']}")
    print(f"  Confianza:                       {resultado['confianza']}")
    print(f"  Fuente:                          {resultado['fuente']}")
    print(f"  Menciones adicionales:           {resultado['menciones_adicionales']}")
    print(f"\n  Final del Full_Report (300 chars):")
    print(f"  ...{full_report[-300:]}")


──────────────────────────────────────────────────────────────────────
Informe idx 3333
──────────────────────────────────────────────────────────────────────
  BI-RADS en columna del dataset:  0
  BI-RADS extraído por el sistema: 0
  Confianza:                       alta
  Fuente:                          bloque_conclusion_estricto
  Menciones adicionales:           []

  Final del Full_Report (300 chars):
  ...reas densas.
- Imágenes nodulares, de naturaleza a determinar, en ambas mamas.
- Calcificaciones con caracteres benignos en ambas mamas.
- BI-RADS ® 0 (según la ACR). Requerirá de estudio complementario.
RECOMENDACIONES:
- Se sugiere correlación con ecografía mamaria para posterior recategorización.

──────────────────────────────────────────────────────────────────────
Informe idx 4177
──────────────────────────────────────────────────────────────────────
  BI-RADS en columna del dataset:  2
  BI-RADS extraído por el sistema: 2
  Confianza:                       alta
  Fuente

In [14]:
# Diagnóstico: ¿qué tiene exactamente el df_clasif para esos índices?
print("=" * 75)
print("DIAGNÓSTICO: ¿qué dice df_clasif vs el dataset original?")
print("=" * 75)

for idx in [3333, 4177]:
    print(f"\n--- idx {idx} ---")
    print(f"\nDesde df_clasif (notebooks/anexos/clasificacion_recomendaciones.csv):")
    fila_clasif = df_clasif.loc[df_clasif.index == idx] if idx in df_clasif.index else None
    if fila_clasif is not None and not fila_clasif.empty:
        print(f"  BI-RADS:      {fila_clasif['BI-RADS'].values[0]}")
        print(f"  categorias:   {fila_clasif['categorias'].values[0]}")
        print(f"  estado_v4:    {fila_clasif['estado_v4'].values[0]}")
        print(f"  severidad_v4: {fila_clasif['severidad_v4'].values[0]}")
    
    print(f"\nDesde df_original (data/processed/reports_cleaned.csv):")
    print(f"  BI-RADS:         {df_original.loc[idx, 'BI-RADS']}")
    print(f"  Recommendations: {df_original.loc[idx, 'Recommendations'][:120]}")

# Verificar la integridad general
print("\n" + "=" * 75)
print("VERIFICACIÓN: ¿los índices de df_clasif coinciden con df_original?")
print("=" * 75)
print(f"  df_clasif filas:   {len(df_clasif)}")
print(f"  df_original filas: {len(df_original)}")
print(f"  df_clasif - índices únicos: {df_clasif.index.nunique()}")
print(f"  df_clasif - rango de índices: [{df_clasif.index.min()}, {df_clasif.index.max()}]")

# Buscar dónde están realmente los 2 casos críticos
print("\n" + "=" * 75)
print("¿Dónde están los REALES BI-RADS 5 con alerta?")
print("=" * 75)
criticos_reales = df_clasif[
    (df_clasif["BI-RADS"] == 5) & 
    (df_clasif["estado_v4"] == "incoherente")
]
print(f"BI-RADS 5 con alerta en df_clasif: {len(criticos_reales)}")
print(f"\nIndices reales:")
print(criticos_reales.index.tolist())

DIAGNÓSTICO: ¿qué dice df_clasif vs el dataset original?

--- idx 3333 ---

Desde df_clasif (notebooks/anexos/clasificacion_recomendaciones.csv):
  BI-RADS:      5
  categorias:   ['estudio_complementario_imagen', 'criterio_medico']
  estado_v4:    incoherente
  severidad_v4: critica

Desde df_original (data/processed/reports_cleaned.csv):
  BI-RADS:         0
  Recommendations: - Se sugiere correlación con ecografía mamaria para posterior recategorización.

--- idx 4177 ---

Desde df_clasif (notebooks/anexos/clasificacion_recomendaciones.csv):
  BI-RADS:      5
  categorias:   ['estudio_complementario_imagen']
  estado_v4:    incoherente
  severidad_v4: critica

Desde df_original (data/processed/reports_cleaned.csv):
  BI-RADS:         2
  Recommendations: - Se sugiere control mamográfico anual.

VERIFICACIÓN: ¿los índices de df_clasif coinciden con df_original?
  df_clasif filas:   4347
  df_original filas: 4357
  df_clasif - índices únicos: 4347
  df_clasif - rango de índices: [0, 4

In [15]:
import pandas as pd

# Cargar ambos archivos
df_clasif = pd.read_csv("anexos/clasificacion_recomendaciones.csv")
df_original = pd.read_csv("../data/processed/reports_cleaned.csv")

print("=" * 75)
print("VERIFICACIÓN DE INTEGRIDAD")
print("=" * 75)

print(f"\nFilas df_clasif:   {len(df_clasif)}")
print(f"Filas df_original: {len(df_original)}")
print(f"\nÍndices df_clasif:   min={df_clasif.index.min()}, max={df_clasif.index.max()}")
print(f"Índices df_original: min={df_original.index.min()}, max={df_original.index.max()}")

# Verificar si los textos de Recommendations coinciden por índice
print("\n¿Las Recommendations coinciden por índice?")
n_match = 0
n_mismatch = 0
for idx in df_clasif.index[:100]:  # primeros 100
    if idx in df_original.index:
        rec_clasif = df_clasif.loc[idx, "Recommendations"]
        rec_orig = df_original.loc[idx, "Recommendations"]
        if str(rec_clasif).strip() == str(rec_orig).strip():
            n_match += 1
        else:
            n_mismatch += 1
print(f"  En los primeros 100 índices: {n_match} coinciden, {n_mismatch} NO coinciden")

# Si NO coinciden, mostrar ejemplo
if n_mismatch > 0:
    for idx in df_clasif.index[:20]:
        if idx in df_original.index:
            rec_clasif = df_clasif.loc[idx, "Recommendations"]
            rec_orig = df_original.loc[idx, "Recommendations"]
            if str(rec_clasif).strip() != str(rec_orig).strip():
                print(f"\n  Ejemplo de mismatch en idx {idx}:")
                print(f"    df_clasif:   {str(rec_clasif)[:120]}")
                print(f"    df_original: {str(rec_orig)[:120]}")
                break

# Ahora buscar el caso idx 3333 en df_clasif usando el TEXTO
print("\n" + "=" * 75)
print("BÚSQUEDA: ¿dónde está REALMENTE el caso idx 3333 de df_clasif en df_original?")
print("=" * 75)
texto_3333_clasif = df_clasif.loc[3333, "Recommendations"]
print(f"Texto que df_clasif asocia a idx 3333: {str(texto_3333_clasif)[:150]}")

# Buscar ese texto en df_original
matches = df_original[df_original["Recommendations"] == texto_3333_clasif]
print(f"\nEse texto aparece en df_original en los índices:")
print(matches.index.tolist()[:10])
if len(matches) > 0:
    print(f"\nPrimer match - BI-RADS: {df_original.loc[matches.index[0], 'BI-RADS']}")

VERIFICACIÓN DE INTEGRIDAD

Filas df_clasif:   4347
Filas df_original: 4357

Índices df_clasif:   min=0, max=4346
Índices df_original: min=0, max=4356

¿Las Recommendations coinciden por índice?
  En los primeros 100 índices: 100 coinciden, 0 NO coinciden

BÚSQUEDA: ¿dónde está REALMENTE el caso idx 3333 de df_clasif en df_original?
Texto que df_clasif asocia a idx 3333: - Se sugiere ecografía mamaria para posterior recategorización y decidir conducta.

Ese texto aparece en df_original en los índices:
[3340]

Primer match - BI-RADS: 5


In [16]:
# Verificar el segundo caso crítico
print("=" * 75)
print("VERIFICACIÓN: caso crítico 2 (df_clasif idx 4177)")
print("=" * 75)

texto_4177_clasif = df_clasif.loc[4177, "Recommendations"]
print(f"\nTexto en df_clasif idx 4177:")
print(f"  {str(texto_4177_clasif)[:200]}")

matches = df_original[df_original["Recommendations"] == texto_4177_clasif]
print(f"\nIndices reales en df_original: {matches.index.tolist()[:10]}")
print(f"Total matches: {len(matches)}")

# Para cada match, mostrar BI-RADS y final del Full_Report
for idx_real in matches.index[:3]:
    print(f"\n--- df_original idx {idx_real} ---")
    print(f"  BI-RADS: {df_original.loc[idx_real, 'BI-RADS']}")
    print(f"  Final Full_Report (200 chars): ...{df_original.loc[idx_real, 'Full_Report'][-200:]}")

VERIFICACIÓN: caso crítico 2 (df_clasif idx 4177)

Texto en df_clasif idx 4177:
  - Se sugiere ecografía mamaria para posterior recategorización.

Indices reales en df_original: [1465, 1466, 1473, 1477, 1482, 1500, 1550, 1568, 1569, 1577]
Total matches: 316

--- df_original idx 1465 ---
  BI-RADS: 0
  Final Full_Report (200 chars): ...alcificaciones con caracteres benignos en ambas mamas.
- Bi-rads 0 (según la ACR). Requerirá de estudio complementario.
RECOMENDACIONES:
- Se sugiere ecografía mamaria para posterior recategorización.

--- df_original idx 1466 ---
  BI-RADS: 0
  Final Full_Report (200 chars): ...alcificaciones con caracteres benignos en ambas mamas.
- Bi-rads 0 (según la ACR). Requerirá de estudio complementario.
RECOMENDACIONES:
- Se sugiere ecografía mamaria para posterior recategorización.

--- df_original idx 1473 ---
  BI-RADS: 0
  Final Full_Report (200 chars): ...alcificaciones con caracteres benignos en ambas mamas.
- Bi-rads 0 (según la ACR). Requerirá de estudio

In [17]:
# Encontrar el ÚNICO BI-RADS 5 con esa recomendación
print("=" * 75)
print("BÚSQUEDA: el verdadero BI-RADS 5 con 'ecografía para recategorización'")
print("=" * 75)

texto_4177 = df_clasif.loc[4177, "Recommendations"]

# Filtrar por TEXTO Y BI-RADS=5
matches_b5 = df_original[
    (df_original["Recommendations"] == texto_4177) &
    (df_original["BI-RADS"] == 5)
]

print(f"\nInformes en df_original con esa recomendación Y BI-RADS=5: {len(matches_b5)}")
print(f"Indices reales: {matches_b5.index.tolist()}")

if len(matches_b5) > 0:
    for idx_real in matches_b5.index:
        print(f"\n--- df_original idx {idx_real} (BI-RADS=5 legítimo) ---")
        print(f"\n  Final Full_Report (500 chars):")
        print(f"  ...{df_original.loc[idx_real, 'Full_Report'][-500:]}")
        
        # Verificar con nuestro extractor
        resultado = extraer_birads(df_original.loc[idx_real, "Full_Report"])
        print(f"\n  Verificación con extractor_birads:")
        print(f"    BI-RADS extraído: {resultado['birads_conclusion']}")
        print(f"    Confianza:        {resultado['confianza']}")
        print(f"    Fuente:           {resultado['fuente']}")

BÚSQUEDA: el verdadero BI-RADS 5 con 'ecografía para recategorización'

Informes en df_original con esa recomendación Y BI-RADS=5: 1
Indices reales: [4187]

--- df_original idx 4187 (BI-RADS=5 legítimo) ---

  Final Full_Report (500 chars):
  ...ón imagen nodular irregular, isodensa y de márgenes espiculados, asociada a calcificaciones groseras heterogéneas.
Imágenes nodulares con caracteres ganglionares en ambas regiones pectoaxilares.
Regiones pectoaxilares ligeramente prominentes, a expensas de tejido adiposo.
No se cuenta con estudios previos para informe comparativo.
CONCLUSIÓN:
- Imagen nodular sospechosa en mama derecha.
- BI-RADS ® 5 (según la ACR).
RECOMENDACIONES:
- Se sugiere ecografía mamaria para posterior recategorización.

  Verificación con extractor_birads:
    BI-RADS extraído: 5
    Confianza:        alta
    Fuente:           bloque_conclusion_estricto
